# Binary classification models

## Dependencies

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
from collections import defaultdict
import pickle
import torch
import time
import seaborn as sns

In [ ]:
from skimage import io
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight
from sklearn.metrics import multilabel_confusion_matrix, f1_score, classification_report

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, Sequential
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input

In [ ]:
random.seed(42)
tf.random.set_seed(42)

## Data

In [ ]:
path_renders = "../capturas/capturas_256x256/classBin"

In [ ]:
database = os.listdir(path_renders)
data_img_labels = database[0]
imgs = database[1:]
print(database)
print(imgs[:10])

In [ ]:
csv_path = os.path.join(path_renders, data_img_labels)

df = pd.read_csv(csv_path, header =0, names=["filename", "label"])

# Añadir ruta completa a cada imagen
df["full_path"] = df["filename"].apply(lambda x: os.path.join(path_renders, x))


print(df[["filename", "correcto"]].head())

In [ ]:
for p in imgs[:10]:    
    img_path = os.path.join(path_renders, p)
    print(img_path)

## Auxiliary functions

In [ ]:
def get_paths_and_labels(df, test_size=0.2):
    df_valid = df[df['full_path'].apply(os.path.exists)].copy()
    
    encoder = LabelEncoder()
    df_valid['label_encoded'] = encoder.fit_transform(df_valid['label'])
    
    paths = df_valid['full_path'].values
    labels = df_valid['label_encoded'].values
    
    # train test (80/20)
    train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
        paths, labels, test_size=test_size, random_state=42, stratify=labels
    )

    # train val (80/20)
    train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.2, random_state=42, stratify=train_val_labels
    )

    return (train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels), encoder


def load_image_tf(path, label, method):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [256, 256])
    
    if method == "resnet":
        img = tf.cast(img, tf.float32) / 255.0
    elif method == "efficientnet":
        img = tf.cast(img, tf.float32) 
    elif method == "mobilenet":
        img = (tf.cast(img, tf.float32) / 127.5) - 1.0
    return img, label

In [ ]:
def plot_from_history(history, epochs):
    hist = history.history if hasattr(history, 'history') else history

    # Determinar cuántas métricas tenemos para ajustar el tamaño de la figura
    has_precision_recall = 'precision' in hist and 'recall' in hist
    rows = 2 if has_precision_recall else 1
    
    plt.figure(figsize=(12, 4 * rows))
    epochs_range = range(len(hist['loss'])) # Usar el largo real de los datos

    # --- 1. Accuracy ---
    plt.subplot(rows, 2, 1)
    plt.plot(epochs_range, hist['accuracy'], label='Train Accuracy')
    plt.plot(epochs_range, hist['val_accuracy'], label='Val Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.legend(loc='lower right')

    # --- 2. Loss ---
    plt.subplot(rows, 2, 2)
    plt.plot(epochs_range, hist['loss'], label='Train Loss')
    plt.plot(epochs_range, hist['val_loss'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.legend(loc='upper right')

    # --- 3. Precision y Recall (Solo si existen) ---
    if has_precision_recall:
        # Precision
        plt.subplot(rows, 2, 3)
        # Nota: Keras a veces nombra la métrica 'precision_1' etc., buscamos la clave
        p_key = [k for k in hist.keys() if 'precision' in k and 'val' not in k][0]
        plt.plot(epochs_range, hist[p_key], label='Train Precision', color='green')
        plt.plot(epochs_range, hist[f'val_{p_key}'], label='Val Precision', color='lightgreen')
        plt.title('Training and Validation Precision')
        plt.xlabel('Epochs')
        plt.legend(loc='lower right')

        # Recall
        plt.subplot(rows, 2, 4)
        r_key = [k for k in hist.keys() if 'recall' in k and 'val' not in k][0]
        plt.plot(epochs_range, hist[r_key], label='Train Recall', color='purple')
        plt.plot(epochs_range, hist[f'val_{r_key}'], label='Val Recall', color='violet')
        plt.title('Training and Validation Recall')
        plt.xlabel('Epochs')
        plt.legend(loc='lower right')

    plt.tight_layout()
    plt.show()

In [ ]:
def save_history(history, nombre_archivo):
    with open(nombre_archivo, "wb") as f:
        pickle.dump(history.history, f)

def load_history(nombre_archivo):
    if os.path.exists(nombre_archivo):
        with open(nombre_archivo, "rb") as f:
            loaded_history = pickle.load(f)
        print(f"Historial cargado desde {nombre_archivo}.")
        return loaded_history
    else:
        print(f"No se encontró el archivo {nombre_archivo}.")
        return None

In [ ]:
def training_pipeline_db(model, train_ds, val_ds, test_ds, patience=3, epochs=10, lr=0.005):
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=patience, verbose=1, min_lr=4e-10
    )

    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]    )

    history = model.fit(
        train_ds, 
        epochs=epochs, 
        validation_data=val_ds, 
        callbacks=[lr_callback]
    )

    print("\nEvaluando en Entrenamiento...")
    train_loss, train_acc = model.evaluate(train_ds, verbose=0)
    print("Evaluando en Test...")
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    
    return test_acc, train_acc, train_loss, test_loss, history

## Get Dataset

In [ ]:
(train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels), encoder = get_paths_and_labels(df, test_size=0.2)

print(encoder.classes_)
for i, nombre in enumerate(encoder.classes_):
    print(f"ID {i} -> Clase: {nombre}")

print(f"Total: {len(train_paths) + len(val_paths) + len(test_paths)}")
print(f"Entrenamiento: {len(train_paths)}")
print(f"Validación:    {len(val_paths)}")
print(f"Test:          {len(test_paths)}")

print("\nDistribución de clases en el dataset completo:")
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='label', data=df, palette='viridis')
plt.title('Distribución de Imágenes por Clase')
plt.xticks(rotation=45)
plt.show()